# Reviewer Revision — Stage A: Verify, Recompute, and Reframe

This notebook addresses the reviewer's first critical issues **before manuscript rewriting**:

1. Recompute Multinomial Naive Bayes on the leakage-clean and original Dreaddit protocols to resolve the Table 3 vs Table 4 contradiction.
2. Recompute exact external dataset counts after deduplication.
3. Compute label-only baselines (majority, all-positive, source-prior random) for every evaluation set.
4. Recompute available Multinomial NB / RoBERTa label-based metrics from the persistent Phase 5 prediction files.
5. Apply Holm-Bonferroni correction to the 15 McNemar comparisons.
6. Save a compact Stage-A audit package to Google Drive.

**Do not edit the paper's results yet.** Use the outputs from this notebook as the canonical audit for the revision.

In [ ]:
!pip -q install datasets openpyxl scipy statsmodels

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import re, hashlib, json, warnings, random, os
import numpy as np
import pandas as pd

from datasets import load_dataset
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, balanced_accuracy_score,
    recall_score, precision_score, confusion_matrix
)
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

SEED = 42
ROOT = Path("/content/reviewer_revision_stageA")
OUT = ROOT / "results"
OUT.mkdir(parents=True, exist_ok=True)

PERSIST = Path("/content/drive/MyDrive/stress_detection_phase5_persistent")
PERSIST_PRED = PERSIST / "predictions"
PERSIST_RESULTS = PERSIST / "results"

REVISION_DRIVE = Path("/content/drive/MyDrive/stress_detection_reviewer_revision")
REVISION_DRIVE.mkdir(parents=True, exist_ok=True)

print("Local output:", OUT)
print("Revision Drive:", REVISION_DRIVE)
print("Phase-5 persistent predictions found:", PERSIST_PRED.exists())

## 1. Reconstruct the exact Dreaddit leakage-clean protocol

In [ ]:
def normalize_text_for_audit(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"https?://\S+|www\.\S+", "<url>", text)
    text = re.sub(r"\s+", " ", text)
    return text

def text_hash(text):
    return hashlib.sha256(normalize_text_for_audit(text).encode("utf-8")).hexdigest()

ds = load_dataset("andreagasparini/dreaddit")
train_raw = ds["train"].to_pandas()[["text","label"]].copy()
test = ds["test"].to_pandas()[["text","label"]].copy()

train_raw["text_hash"] = train_raw["text"].map(text_hash)
test["text_hash"] = test["text"].map(text_hash)

overlap_hashes = set(train_raw["text_hash"]) & set(test["text_hash"])

clean_train = train_raw.drop_duplicates("text_hash", keep="first").copy()
clean_train = clean_train[~clean_train["text_hash"].isin(overlap_hashes)].reset_index(drop=True)

print("Original train:", len(train_raw))
print("Leakage-clean train:", len(clean_train))
print("Official test:", len(test))
print("Train duplicate rows:", train_raw.duplicated("text_hash").sum())
print("Normalized train-test overlaps:", len(overlap_hashes))
print("\nClean class counts:")
print(clean_train.label.value_counts().sort_index())
print("\nTest class counts:")
print(test.label.value_counts().sort_index())

## 2. Resolve Table 3 vs Table 4 by rerunning MNB under both protocols

This reproduces the original Phase-2 search space rather than hard-coding the expected answer.

In [ ]:
CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

mnb_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True, ngram_range=(1,2), min_df=2, max_df=0.98
    )),
    ("clf", MultinomialNB())
])

mnb_grid = {
    "tfidf__max_features": [10000, 20000],
    "clf__alpha": [0.25, 0.5, 1.0]
}

def run_mnb_protocol(train_df, protocol):
    search = GridSearchCV(
        mnb_pipe, mnb_grid, scoring="f1_macro", cv=CV,
        n_jobs=-1, refit=True
    )
    search.fit(train_df["text"].astype(str), train_df["label"].astype(int))
    model = search.best_estimator_

    Xte = test["text"].astype(str)
    y = test["label"].astype(int).to_numpy()
    pred = model.predict(Xte)
    p_pos = model.predict_proba(Xte)[:,1]

    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()

    row = {
        "protocol": protocol,
        "train_n": len(train_df),
        "best_cv_macro_f1": search.best_score_,
        "best_params": json.dumps(search.best_params_),
        "accuracy": accuracy_score(y,pred),
        "macro_f1": f1_score(y,pred,average="macro",zero_division=0),
        "roc_auc": roc_auc_score(y,p_pos),
        "balanced_accuracy": balanced_accuracy_score(y,pred),
        "positive_recall": recall_score(y,pred,pos_label=1,zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    }

    pred_df = pd.DataFrame({
        "text": Xte.reset_index(drop=True),
        "true_label": y,
        "predicted_label": pred,
        "p_pos": p_pos
    })
    pred_df.to_csv(OUT/f"MNB_{protocol}_Dreaddit_Test.csv", index=False)
    pred_df.to_csv(REVISION_DRIVE/f"MNB_{protocol}_Dreaddit_Test.csv", index=False)

    return row, model

rows = []
models = {}

r, models["clean"] = run_mnb_protocol(clean_train, "leakage_clean")
rows.append(r)

r, models["original"] = run_mnb_protocol(train_raw.reset_index(drop=True), "original")
rows.append(r)

mnb_protocols = pd.DataFrame(rows)
display(mnb_protocols.round(6))

mnb_protocols.to_csv(OUT/"mnb_protocol_audit.csv", index=False)
mnb_protocols.to_csv(REVISION_DRIVE/"mnb_protocol_audit.csv", index=False)

print("\nIMPORTANT: Use the leakage_clean row as the canonical Table 3/Table 4 MNB value.")

## 3. Reconstruct the cleaned external datasets and exact class counts

In [ ]:
import requests

RAW = ROOT / "raw"
RAW.mkdir(exist_ok=True)

FILES = {
    "Reddit_Title":"Reddit_Title.xlsx",
    "Reddit_Combi":"Reddit_Combi.xlsx",
    "Twitter_Full":"Twitter_Full.xlsx",
    "Twitter_NonAdvert":"Twitter_Non-Advert.xlsx",
}
BASE = "https://raw.githubusercontent.com/SenticNet/stress-detection/master"

def prep(df):
    df = df.copy()
    df["text"] = df["text"].astype("string").str.strip()
    df.loc[df["text"]=="","text"] = pd.NA
    df["label"] = pd.to_numeric(df["label"],errors="coerce")
    df = df.dropna(subset=["text","label"]).copy()
    df["label"] = df["label"].astype(int)
    df["text_hash"] = df["text"].map(text_hash)
    return df

external = {}
audit_rows = []

for name, fn in FILES.items():
    p = RAW/fn
    if not p.exists():
        r = requests.get(f"{BASE}/{fn}", timeout=60)
        r.raise_for_status()
        p.write_bytes(r.content)

    raw = pd.read_excel(p)

    if name=="Reddit_Title":
        text, label = raw["title"], raw["label"]
    elif name=="Reddit_Combi":
        text = (raw["title"].fillna("").astype(str).str.strip() + " " +
                raw["body"].fillna("").astype(str).str.strip()).str.strip()
        label = raw["label"]
    elif name=="Twitter_Full":
        text, label = raw["text"], raw["labels"]
    else:
        text, label = raw["text"], raw["label"]

    df0 = prep(pd.DataFrame({"text":text,"label":label}))
    original_n = len(df0)

    # Count rows in duplicate same-label groups (rows removed by retaining one copy)
    group_sizes = df0.groupby("text_hash").size()
    label_nunique = df0.groupby("text_hash")["label"].nunique()
    conflicts = set(label_nunique[label_nunique>1].index)

    conflict_rows = int(df0["text_hash"].isin(conflicts).sum())
    conflict_groups = len(conflicts)

    nonconf = df0[~df0["text_hash"].isin(conflicts)].copy()
    duplicate_rows_removed = int(len(nonconf) - len(nonconf.drop_duplicates("text_hash", keep="first")))

    primary = nonconf.drop_duplicates("text_hash", keep="first").reset_index(drop=True)
    external[name] = primary

    audit_rows.append({
        "dataset": name,
        "original_rows": original_n,
        "duplicate_rows_removed": duplicate_rows_removed,
        "conflict_rows_removed": conflict_rows,
        "conflict_groups": conflict_groups,
        "primary_rows": len(primary),
        "primary_negative": int((primary.label==0).sum()),
        "primary_positive": int((primary.label==1).sum()),
    })

audit = pd.DataFrame(audit_rows)
display(audit)

audit.to_csv(OUT/"external_dataset_audit_corrected.csv", index=False)
audit.to_csv(REVISION_DRIVE/"external_dataset_audit_corrected.csv", index=False)

## 4. Compute label-only baselines

The source-prior random baseline uses the positive prevalence of the leakage-clean Dreaddit training set and averages Macro-F1 across 1,000 draws.

In [ ]:
source_prior_pos = clean_train["label"].mean()
rng = np.random.default_rng(42)

eval_sets = {
    "Dreaddit_Test": test.reset_index(drop=True),
    **external
}

def trivial_baselines(y, source_prior_pos, n_draws=1000):
    y = np.asarray(y, dtype=int)
    majority_label = np.bincount(y).argmax()

    maj = np.full(len(y), majority_label, dtype=int)
    all_pos = np.ones(len(y), dtype=int)

    rand_vals = []
    for _ in range(n_draws):
        pred = rng.binomial(1, source_prior_pos, size=len(y))
        rand_vals.append(f1_score(y,pred,average="macro",zero_division=0))

    return {
        "majority_macro_f1": f1_score(y,maj,average="macro",zero_division=0),
        "all_positive_macro_f1": f1_score(y,all_pos,average="macro",zero_division=0),
        "source_prior_random_macro_f1_mean": float(np.mean(rand_vals)),
        "best_label_only_macro_f1": max(
            f1_score(y,maj,average="macro",zero_division=0),
            f1_score(y,all_pos,average="macro",zero_division=0),
            float(np.mean(rand_vals))
        ),
        "n": len(y),
        "negative": int((y==0).sum()),
        "positive": int((y==1).sum()),
        "positive_prevalence": float((y==1).mean())
    }

baseline_rows = []
for name, df in eval_sets.items():
    row = {"dataset":name, **trivial_baselines(df["label"], source_prior_pos)}
    baseline_rows.append(row)

baselines = pd.DataFrame(baseline_rows)
display(baselines.round(6))

baselines.to_csv(OUT/"label_only_baselines.csv", index=False)
baselines.to_csv(REVISION_DRIVE/"label_only_baselines.csv", index=False)

print("\nPay special attention to Twitter_Full and Twitter_NonAdvert.")

## 5. Recompute label-based metrics from persistent Phase-5 NB/RoBERTa predictions

This step uses the exact predictions underlying the bootstrap/McNemar analysis. If the persistent folder is unavailable, the notebook will report which files are missing.

In [ ]:
def label_metrics(df):
    y = df["true_label"].to_numpy()
    p = df["predicted_label"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y,p,labels=[0,1]).ravel()
    return {
        "accuracy": accuracy_score(y,p),
        "macro_f1": f1_score(y,p,average="macro",zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y,p),
        "macro_precision": precision_score(y,p,average="macro",zero_division=0),
        "macro_recall": recall_score(y,p,average="macro",zero_division=0),
        "positive_recall": recall_score(y,p,pos_label=1,zero_division=0),
        "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp)
    }

metric_rows = []
missing = []

for dataset in eval_sets:
    nb_path = PERSIST_PRED/f"NB_{dataset}.csv"
    if nb_path.exists():
        d = pd.read_csv(nb_path)
        metric_rows.append({"dataset":dataset,"model":"Multinomial NB","seed":np.nan,**label_metrics(d)})
    else:
        missing.append(str(nb_path))

    for seed in [42,52,62]:
        p = PERSIST_PRED/f"RB_{seed}_{dataset}.csv"
        if p.exists():
            d = pd.read_csv(p)
            metric_rows.append({"dataset":dataset,"model":"RoBERTa-base","seed":seed,**label_metrics(d)})
        else:
            missing.append(str(p))

metrics = pd.DataFrame(metric_rows)

if len(metrics):
    display(metrics.round(6))
    metrics.to_csv(OUT/"phase5_label_metrics_recomputed.csv", index=False)
    metrics.to_csv(REVISION_DRIVE/"phase5_label_metrics_recomputed.csv", index=False)
else:
    print("No persistent prediction files were found.")

if missing:
    print("\nMissing files:", len(missing))
    print("\n".join(missing[:20]))

## 6. Holm-Bonferroni correction for the 15 McNemar tests

In [ ]:
mcnemar_path = PERSIST_RESULTS/"mcnemar.csv"

if mcnemar_path.exists():
    mc = pd.read_csv(mcnemar_path)
    reject, p_adj, _, _ = multipletests(mc["p_value"].to_numpy(), alpha=0.05, method="holm")
    mc["holm_adjusted_p"] = p_adj
    mc["holm_significant_0_05"] = reject

    display(mc.round(8))
    print("Largest Holm-adjusted p:", mc["holm_adjusted_p"].max())
    print("All 15 significant after Holm:", bool(mc["holm_significant_0_05"].all()))

    mc.to_csv(OUT/"mcnemar_holm_corrected.csv", index=False)
    mc.to_csv(REVISION_DRIVE/"mcnemar_holm_corrected.csv", index=False)
else:
    print("mcnemar.csv not found at:", mcnemar_path)

## 7. Build the Stage-A audit summary

In [ ]:
summary_rows = []

# Canonical MNB contradiction resolution
clean_row = mnb_protocols[mnb_protocols.protocol=="leakage_clean"].iloc[0]
orig_row = mnb_protocols[mnb_protocols.protocol=="original"].iloc[0]

summary_rows.append({
    "issue":"MNB Table 3 vs Table 4",
    "result":f"Leakage-clean: accuracy={clean_row.accuracy:.4f}, Macro-F1={clean_row.macro_f1:.4f}, AUC={clean_row.roc_auc:.4f}; "
             f"Original: accuracy={orig_row.accuracy:.4f}, Macro-F1={orig_row.macro_f1:.4f}, AUC={orig_row.roc_auc:.4f}",
    "manuscript_action":"Use the recomputed leakage-clean values everywhere and report the original protocol separately."
})

# Twitter baselines
for dsname in ["Twitter_Full","Twitter_NonAdvert"]:
    r = baselines[baselines.dataset==dsname].iloc[0]
    summary_rows.append({
        "issue":f"{dsname} label-only baseline",
        "result":f"Majority={r.majority_macro_f1:.4f}; all-positive={r.all_positive_macro_f1:.4f}; "
                 f"source-prior random={r.source_prior_random_macro_f1_mean:.4f}; best={r.best_label_only_macro_f1:.4f}",
        "manuscript_action":"Add to Table 6 and revised Figure 2; reframe transfer relative to this baseline."
    })

if mcnemar_path.exists():
    summary_rows.append({
        "issue":"Multiple comparisons",
        "result":f"All Holm significant={bool(mc.holm_significant_0_05.all())}; "
                 f"largest adjusted p={mc.holm_adjusted_p.max():.6g}",
        "manuscript_action":"Report Holm-adjusted McNemar p-values and distinguish sampling from seed variability."
    })

summary = pd.DataFrame(summary_rows)
display(summary)

summary.to_csv(OUT/"STAGE_A_AUDIT_SUMMARY.csv", index=False)
summary.to_csv(REVISION_DRIVE/"STAGE_A_AUDIT_SUMMARY.csv", index=False)

print("\nSaved all Stage-A outputs to:")
print(REVISION_DRIVE)

## Send back these outputs

After the notebook finishes, send screenshots (or download/upload the CSVs) for:

1. `mnb_protocol_audit.csv`
2. `external_dataset_audit_corrected.csv`
3. `label_only_baselines.csv`
4. `phase5_label_metrics_recomputed.csv` (if found)
5. `mcnemar_holm_corrected.csv`
6. `STAGE_A_AUDIT_SUMMARY.csv`

Once these are verified, we will update Tables 2–4 and the statistical wording, then move to **Stage B: all-six-model external transfer and diagnostics**.